# Lab 14. Deploy practice.

Training a model is only half the job. A model that lives only in a Jupyter notebook creates zero business value. As soon as you hand off results to a product team, a backend engineer, or a data pipeline — you need a running, queryable service. Modern ML roles expect you to own the full lifecycle of your solution:


<img src="https://media.licdn.com/dms/image/v2/D4D22AQGeKMIBR7I_UQ/feedshare-shrink_800/B4DZpc_rxIGgAg-/0/1762496887429?e=2147483647&v=beta&t=RdoZwXObOERyyOvhsyqxrLMC1wLIVxffHg-xaXzm-0s" width="400"/>

There exists different kind of solution management depending upon your architecture, tasks, integration, capacity, etc. But you can notice that every system management requires deployment which is basically our final goal in real-world problems.

In this lab we will focus on deployment stage using most viable and simple stack for simple AI-agents based on one-model, so no ReACT, RAG, Multi-agents. Before swithcing to the code part, let's start from simple concepts.

**Kubernetes (k8s/k3s)** is an open-source container orchestration system. It automatically manages:

- **deployment** of containers
- **scaling** and **self-healing** of containerized applications.

Here are its fundamental building blocks:

- **Pod** - Smallest deployable unit. Runs 1+ containers sharing network & storage. Ephemeral by design.

 - **Node** - Physical or virtual machine that runs pods. Has CPU/GPU, RAM, and disk. Managed by control plane.

- **Cluster** - Set of nodes managed together. Has 1 control plane + N worker nodes (not manadatory).

- **Deployment** - Declares desired state: how many pod replicas, which image, resource limits.

- **Service** - Stable network endpoint for pods. Load-balances across replicas. Types: ClusterIP, NodePort, LoadBalancer.

- **Namespace** - Virtual cluster inside a cluster. Isolates resources by team, env, or project.

- **ConfigMap / Secret** - Externalize config and credentials from container images. Mounted as env vars or files.

- **PersistentVolume** - Durable storage that outlives pods. Critical for storing large model weights.



## Let's start with arranging our cluster

### Update package index and upgrade existing packages

```
sudo apt-get update && sudo apt-get upgrade -y
```

# Install build essentials and dependencies
```
sudo apt-get install -y \
  build-essential \
  curl wget git vim htop \
  python3.11 python3.11-dev python3-pip \
  ca-certificates gnupg lsb-release \
  software-properties-common apt-transport-https
```

Next we need to install **Docker** and **contaierd**. You might ask why do we need both as we want kubernetes to be used as orchestrator. Well, the answer is no that obvious, they have its own role in deployment stage.

- Docker is usually a developer-facing tool for building and pushing images.
- Containerd is a runtime that runs containers inside Kubernetes.

In production, Kubernetes never talks to Docker at all — it talks directly to containerd.

## Install Docker Engine
```
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg
```

```
echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] \
  https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
```
```
sudo apt-get update
sudo apt-get install -y docker-ce docker-ce-cli containerd.io
sudo usermod -aG docker $USER
```
### Install NVIDIA Container Toolkit (allows GPUs in containers)
```
distribution=$(. /etc/os-release; echo $ID$VERSION_ID)
```
```
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
```
```
curl -s -L https://nvidia.github.io/libnvidia-container/$distribution/libnvidia-container.list | \
  sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \
  sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list
```

```
sudo apt-get update
```

Sometimes necessary proceeses remain in hold status. We should unhold them

```
sudo apt-mark unhold libnvidia-container1
sudo apt-mark unhold libnvidia-container-tools
```

And proceed with installing nvidi toolkit

```
sudo apt-get install -y nvidia-container-toolkit
sudo nvidia-ctk runtime configure --runtime=docker
sudo systemctl restart docker
```

Verify GPU is accessible in Docker

```
docker run --rm --gpus all nvidia/cuda:12.1.0-base-ubuntu22.04 nvidia-smi
```

You should get something like this

```
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               On  |   00000000:06:00.0 Off |                  Off |
| 30%   34C    P8              8W /  230W |       1MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+
```

## Install kubectl & kubeadm

```
curl -LO "https://dl.k8s.io/release/$(curl -L -s https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
sudo install -o root -g root -m 0755 kubectl /usr/local/bin/kubectl
```

Check that kubectl is installed

```
kubectl version --client
```

For local dev install k3s (lightweight K8s — single command!)

```
curl -sfL https://get.k3s.io | sh -
```

Now we need to apply config to see the status of k3s nodes and pods

```
sudo chmod 644 /etc/rancher/k3s/k3s.yaml
export KUBECONFIG=/etc/rancher/k3s/k3s.yaml
```

Verify cluster is running

```
kubectl get nodes
```

## Install NVIDIA Device Plugin for Kubernetes
This plugin makes GPUs visible to Kubernetes as schedulable resources (nvidia.com/gpu)

Deploy NVIDIA device plugin daemonset

```
kubectl create -f https://raw.githubusercontent.com/NVIDIA/k8s-device-plugin/v0.14.3/nvidia-device-plugin.yml
```

We will change this DaemonSet. Sometimes it does not make our GPU visible.

```
cat > nvidia-device-plugin-fixed.yml << 'EOF'
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: nvidia-device-plugin-daemonset
  namespace: kube-system
spec:
  selector:
    matchLabels:
      name: nvidia-device-plugin-ds
  updateStrategy:
    type: RollingUpdate
  template:
    metadata:
      labels:
        name: nvidia-device-plugin-ds
    spec:
      tolerations:
        - key: nvidia.com/gpu
          operator: Exists
          effect: NoSchedule
      priorityClassName: system-node-critical
      runtimeClassName: nvidia
      containers:
        - image: nvcr.io/nvidia/k8s-device-plugin:v0.17.1
          name: nvidia-device-plugin-ctr
          env:
            - name: FAIL_ON_INIT_ERROR
              value: "false"
          securityContext:
            allowPrivilegeEscalation: false
            capabilities:
              drop: ["ALL"]
          volumeMounts:
            - name: device-plugin
              mountPath: /var/lib/kubelet/device-plugins
      volumes:
        - name: device-plugin
          hostPath:
            path: /var/lib/kubelet/device-plugins
EOF
```
In some cases it is easier to create runtime manually and then add it to DaemonSet directly. It can save your time. I have a story that because of this I lost **a week** of time to make it work properly.

Adding runtime (if necessary)...

```
kubectl apply -f - <<EOF
apiVersion: node.k8s.io/v1
kind: RuntimeClass
metadata:
  name: nvidia
handler: nvidia
EOF
```

Before launch let's make runtime for contaierd


```
sudo nvidia-ctk runtime configure --runtime=containerd
sudo systemctl restart containerd
```

Launch it

```
kubectl apply -f nvidia-device-plugin-fixed.yml
```

Check that gpu is available

```
kubectl label nodes $(kubectl get nodes -o name | head -1 | cut -d/ -f2) \
  accelerator=nvidia-gpu
```

OR

```
kubectl get nodes -o json | jq '.items[].status.allocatable'
```

Let's check our node IP

```
kubectl get nodes -o wide
```

Next we need a place where to preserve our image with model. We can use a basic docker registry

```
docker run -d \
  --name registry \
  --restart always \
  -p 5000:5000 \
  -v /opt/registry:/var/lib/registry \
  registry:2
```
Next couple manipulations to make our registry visible for kubernets node

```
cat << EOF | sudo tee /etc/rancher/k3s/registries.yaml
mirrors:
  "${SERVER_IP}:5000":
    endpoint:
      - "http://${SERVER_IP}:5000"
EOF
```
Verify the file looks right

```
cat /etc/rancher/k3s/registries.yaml
```

Restart our kuber cluster

```
sudo systemctl restart k3s
```

Add insecure registry to Docker daemon config

```
SERVER_IP=$(hostname -I | awk '{print $1}')

sudo tee /etc/docker/daemon.json << EOF
{
  "insecure-registries": ["${SERVER_IP}:5000"]
}
EOF
```
Sometimes it causes fail, then delete daemon.json
```
sudo rm -f /etc/docker/daemon.json
```
And restart

```
sudo systemctl restart docker
```

Repeat one more time...

## All necessary modules for our AI agent based on Gigachat

OK, so now let's try to build our image based on GigaChat model. We need the following files:

BUT: first create two folders:

- where files for image will be preserved
- deployment configuration for kubernetes will be located

```
mkdir LLM_kuber
mkdir agent
cd agent
mkdir app
```
We need at least 4 files:
- requirements.txt - for all necessary libraries for our image
- model.py - where we initiate our model
- main.py - where we define our API and request schema
- Dockerfile - where we define configuration of image with our LLM
- deployment.yaml - deployment configuration for kubernetes

NOW you should arrange files as following:
```
├── agent/
│    ├── requirements.txt
│    ├── Dockerfile
│    ├── app
│       │── model.py
|       │── main.py
├── LLM_kuber/
│    ├── deployment.yaml

```

# main.py

```
nano main.py
```
```
import os
import time
import logging
import traceback
import asyncio
import torch

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from app.model import model, tokenizer, MODEL_NAME


class HealthCheckFilter(logging.Filter):
    def filter(self, record):
        return "/health" not in record.getMessage()

logging.getLogger("uvicorn.access").addFilter(HealthCheckFilter())


class GenerateRequest(BaseModel):
    prompt: str
    system_prompt: str  = "Ты — полезный ИИ-ассистент. Отвечай точно и лаконично."
    max_new_tokens: int = 512
    temperature: float  = 0.6


app = FastAPI(title="GigaChat3 Inference API", version="2.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model":  MODEL_NAME,
        "device": str(model.device),
    }


@app.post("/generate")
async def generate(req: GenerateRequest):
    messages = [
        {"role": "system", "content": req.system_prompt},
        {"role": "user",   "content": req.prompt},
    ]

    # Tokenize prompt as plain string first, then encode to tensor
    # This matches the exact pattern from the working GigaChat screenshot
    prompt_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,          # returns a plain string
    )

    # Now tokenize the string — returns a BatchEncoding with .input_ids
    inputs = tokenizer(prompt_text, return_tensors="pt")
    input_ids = inputs["input_ids"].to(model.device)
    prompt_len = input_ids.shape[1]

    t0 = time.perf_counter()

    def _infer():
        with torch.inference_mode():
            outputs = model.generate(
                input_ids,
                max_new_tokens=req.max_new_tokens,
                do_sample=req.temperature > 0,
                temperature=req.temperature if req.temperature > 0 else None,
                top_p=0.95,
            )
        return tokenizer.decode(
            outputs[0][prompt_len:],
            skip_special_tokens=True,
        )

    try:
        result = await asyncio.get_event_loop().run_in_executor(None, _infer)
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=repr(e))

    return {
        "text":             result,
        "tokens_generated": len(tokenizer.encode(result)),
        "latency_ms":       round((time.perf_counter() - t0) * 1000, 1),
    }


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
```

## model.py
```
nano model.py
```
```
import os
import logging
import warnings
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig


def _fix_routed_scaling_factor() -> None:
    import huggingface_hub.dataclasses as hf_dc
    _original = hf_dc._validate_simple_type
    def _patched(name, value, expected_type):
        if name == "routed_scaling_factor" and expected_type is float and isinstance(value, int):
            return
        return _original(name, value, expected_type)
    hf_dc._validate_simple_type = _patched


_fix_routed_scaling_factor()

# Suppress transformers load reports (UNEXPECTED / MISSING / CONVERSION)
transformers.logging.set_verbosity_error()

# Suppress general Python warnings from transformers / accelerate
warnings.filterwarnings("ignore")

MODEL_NAME = os.getenv("MODEL_ID", "ai-sage/GigaChat3-10B-A1.8B-bf16")
CACHE_DIR  = os.getenv("MODEL_CACHE_DIR", "/mnt/models")
# HF_TOKEN   = os.getenv("HF_TOKEN",         None)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=CACHE_DIR,
    # token=HF_TOKEN,
)

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=CACHE_DIR,
    # token=HF_TOKEN,
)
model.generation_config = GenerationConfig.from_pretrained(
    MODEL_NAME,
    cache_dir=CACHE_DIR,
    # token=HF_TOKEN,
)
model.eval()

print(f"Model ready on {model.device}")
```

## requirements.txt
```
torch==2.4.1
torchvision
torchaudio

# HuggingFace stack — versions confirmed working with GigaChat3
transformers>=4.47.0
tokenizers>=0.20.0
accelerate>=0.34.0
huggingface_hub>=0.26.0
sentencepiece
protobuf

# API
fastapi==0.111.0
uvicorn[standard]==0.29.0
pydantic==2.7.1

# Client / utilities
httpx
python-dotenv
```
## Dockerfile

```
FROM nvidia/cuda:12.1.1-cudnn8-runtime-ubuntu22.04

ENV DEBIAN_FRONTEND=noninteractive \
    PYTHONUNBUFFERED=1 \
    PIP_NO_CACHE_DIR=1 \
    MODEL_CACHE_DIR=/mnt/models

RUN apt-get update && apt-get install -y \
    python3.11 python3.11-dev python3-pip curl \
    && rm -rf /var/lib/apt/lists/*

RUN update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1

WORKDIR /app

COPY requirements.txt .

# Install in exact order to avoid NumPy 2.x conflicts:
# 1. numpy<2  2. torch with CUDA  3. everything else
RUN pip3 install --upgrade pip && \
    pip3 install "numpy<2" && \
    pip3 install torch==2.4.1 torchvision torchaudio \
        --index-url https://download.pytorch.org/whl/cu121 && \
    pip3 install -r requirements.txt

COPY app/ ./app/

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=10s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
```

## deployment.yaml
```
---
apiVersion: v1
kind: Namespace
metadata:
  name: ml-serving

---
apiVersion: v1
kind: Secret
metadata:
  name: llm-secrets
  namespace: ml-serving
type: Opaque
stringData:
  api-key: "dev-secret"
  hf-token: ""           # GigaChat3 is public — leave empty

---
apiVersion: v1
kind: ConfigMap
metadata:
  name: llm-config
  namespace: ml-serving
data:
  model_id: "ai-sage/GigaChat3-10B-A1.8B-bf16"

---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: model-cache-pvc
  namespace: ml-serving
spec:
  accessModes:
    - ReadWriteOnce
  storageClassName: local-path
  resources:
    requests:
      storage: 30Gi      # GigaChat3 bf16 ~20 GB

---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: gigachat-agent
  namespace: ml-serving
  labels:
    app: gigachat-agent
spec:
  replicas: 1
  selector:
    matchLabels:
      app: gigachat-agent
  template:
    metadata:
      labels:
        app: gigachat-agent
    spec:
      runtimeClassName: nvidia
      nodeSelector:
        accelerator: nvidia-gpu
      containers:
        - name: gigachat-api
          image: 185.182.108.95:5000/gigachat-agent:latest
          imagePullPolicy: Always
          ports:
            - containerPort: 8000
          env:
            - name: API_KEY
              valueFrom:
                secretKeyRef:
                  name: llm-secrets
                  key: api-key
            - name: HF_TOKEN
              valueFrom:
                secretKeyRef:
                  name: llm-secrets
                  key: hf-token
            - name: MODEL_ID
              valueFrom:
                configMapKeyRef:
                  name: llm-config
                  key: model_id
            - name: MODEL_CACHE_DIR
              value: "/mnt/models"
          resources:
            requests:
              memory: "16Gi"
              cpu: "4"
              nvidia.com/gpu: "1"
            limits:
              memory: "28Gi"
              cpu: "8"
              nvidia.com/gpu: "1"
          volumeMounts:
            - name: model-cache
              mountPath: /mnt/models
          # Model loads at startup — give it time before routing traffic
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 120
            periodSeconds: 10
            failureThreshold: 12    # 2 min grace period total
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 180
            periodSeconds: 20
            failureThreshold: 3
      volumes:
        - name: model-cache
          persistentVolumeClaim:
            claimName: model-cache-pvc

---
apiVersion: v1
kind: Service
metadata:
  name: gigachat-svc
  namespace: ml-serving
spec:
  selector:
    app: gigachat-agent
  type: NodePort
  ports:
    - protocol: TCP
      port: 80
      targetPort: 8000
      nodePort: 30800
```

Let's build and push our image to our created registry

```
SERVER_IP=$(hostname -I | awk '{print $1}')

# Build
docker build -t gigachat-agent:latest .

# Tag
docker tag gigachat-agent:latest localhost:5000/gigachat-agent:latest

# Push
docker push localhost:5000/gigachat-agent:latest

Let's deploy from LLM_kuber folder

```
kubectl apply -f deployment.yaml

You can check the status of pod
```
kubectl describe pod gigachat-agent-5cc765d7ff-pgtll -n=ml-serving
```

You can delete your daemonset if something goes wrong...
```
kubectl delete deployment gigachat-agent -n=ml-serving
```

## Final result
After checking log of your pod
```
kubectl logs gigachat-agent-cd75577dd-j2l66 -n=ml-serving
```

you should see something like this in your terminal:
```
CUDA available: True
GPU: NVIDIA RTX A5000
Loading tokenizer: ai-sage/GigaChat3-10B-A1.8B-bf16
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading model: ai-sage/GigaChat3-10B-A1.8B-bf16
Loading weights: 100%|██████████| 363/363 [00:05<00:00, 62.33it/s]
Model ready on cuda:0
INFO:     Started server process [1]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
```

Wait until pod will be fully ready - should be 1/1
```
ml-serving    gigachat-agent-5cc765d7ff-pgtll           1/1     Running     1 (2m18s ago)   7m50s
```

# Requests

Now you can send requests, but before you need to Port-forward so you can reach it from the server terminal
```
kubectl port-forward -n ml-serving svc/gigachat-svc 8000:80 &
sleep 2
```

### Health check
```
curl http://localhost:8000/health
```
You should get something like this
```
{"status":"ok","model":"ai-sage/GigaChat3-10B-A1.8B-bf16","device":"cuda:0"}
```
Send request:
```
curl -s -X POST "http://localhost:8000/generate?text_only=true"   -H "Content-Type: application/json"   -d '{"prompt": "Что такое Kubernetes?", "max_new_tokens": 100, "temperature": 0}'
```

Delete port-forward if it is neede later...
```
pkill -f port-forward
```

## External requests

Now you can use your model in service you want, just install httpx to process request from Internet

In [1]:
!pip install httpx

In [5]:
import httpx, os

os.environ["LLM_URL"]     = "http://185.182.108.108:30800"
os.environ["LLM_API_KEY"] = "dev-secret"

def generate(prompt, max_new_tokens=200, temperature=0.6):
    r = httpx.post(
        f"{os.environ['LLM_URL']}/generate",
        headers={"X-API-Key": os.environ["LLM_API_KEY"]},
        json={"prompt": prompt, "max_new_tokens": max_new_tokens, "temperature": temperature},
        timeout=120,
    )
    r.raise_for_status()
    data = r.json()
    print(f"Latency: {data['latency_ms']} ms | Tokens: {data['tokens_generated']}")
    return data["text"]

# Test it
print(generate("Расскажи коротко что такое kubernetes?"))

Latency: 2128.1 ms | Tokens: 44
Kubernetes — это система для автоматизации развертывания, масштабирования и управления контейнерами приложений. Она позволяет группировать контейнеры в кластеры, управлять ими как единым целым и обеспечивает высокую отказоустойчивость.
